In [4]:
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. עיצוב CSS מותאם ליישור לימין
display(HTML("""
<style>
    .rtl-checklist {
        direction: rtl !important;
        text-align: right !important;
    }
    .rtl-checklist .widget-checkbox {
        flex-direction: row-reverse !important;
        justify-content: flex-end !important;
    }
    .rtl-checklist .widget-checkbox label {
        margin-right: 8px !important;
        margin-left: 0px !important;
    }
</style>
"""))

# 2. הגדרת המשימות והשלבים
tasks_data = [
    # חלק 1
    ("--- 1. הכנת הנתונים והסביבה ---", "header"),
    ("התקנת הספריות הנדרשות לפיתוח", "task"),
    ("הורדת ה-Dataset מ-Kaggle", "task"),
    ("טיפול בערכים חסרים בדאטה", "task"),
    ("בחירת 5–7 פיצ'רים + עמודת Target", "task"),
    
    # חלק 2
    ("--- 2. אימון המודל וה-Pipeline ---", "header"),
    ("המרת משתנים קטגוריאליים לערכים נומריים", "task"),
    ("פיצול הנתונים ל-Train ו-Test", "task"),
    ("בניית Pipeline המשולב StandardScaler ו-SVC", "task"),
    ("אימון ה-Pipeline על נתוני האימון (fit)", "task"),
    ("חישוב מטריקות הערכה (Accuracy, Confusion Matrix, Classification Report)", "task"),
    ("שמירת ה-Pipeline המאומן לקובץ", "task"),
    
    # חלק 3
    ("--- 3. שרת FastAPI ---", "header"),
    ("הקמת תשתית השרת ב-FastAPI", "task"),
    ("מנגנון טעינת קובץ המודל בעליית השרת (כולל טיפול במקרה של קובץ חסר)", "task"),
    ("יצירת Endpoint להחזרת נתוני ומטריקות המודל (/model/info)", "task"),
    ("יצירת Endpoint לקבלת נתוני משתמש והחזרת חיזוי (/predict)", "task"),
    
    # חלק 4
    ("--- 4. ממשק המשתמש (HTML / Frontend) ---", "header"),
    ("בניית דף נתוני המודל והצגת המטריקות דרך ה-API", "task"),
    ("בניית דף טופס קליטת נתונים מהמשתמש", "task"),
    ("חיבור כפתור ה-Check ל-API להרצת החיזוי", "task"),
    ("הצגת תוצאת החיזוי הוויזואלית (מאושר / לא מאושר)", "task"),
    
    # חלק 5
    ("--- 5. איכות, בדיקות והגשה ---", "header"),
    ("בדיקת התנהגות המערכת כשקובץ המודל חסר", "task"),
    ("כתיבת תיעוד (Docstrings) בקוד", "task"),
    ("הוספת לוגים (רשות - בונוס)", "task"),
    ("העלאת האפליקציה ל-Render (רשות - בונוס)", "task"),
    ("העלאת הקוד והמודל ל-GitHub", "task"),
]

checkboxes = []
ui_elements = []

# 3. יצירת רכיבי הממשק
for item, item_type in tasks_data:
    if item_type == "header":
        ui_elements.append(widgets.HTML(value=f"<h4 style='margin-top:15px; margin-bottom:5px; text-align:right;'>{item}</h4>"))
    else:
        cb = widgets.Checkbox(value=False, description=item, indent=False, layout=widgets.Layout(width='auto'))
        checkboxes.append(cb)
        ui_elements.append(cb)

# מד התקדמות
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(checkboxes),
    description='התקדמות:',
    bar_style='info',
    orientation='horizontal',
    layout=widgets.Layout(width='300px')
)

progress_label = widgets.Label(value=f"0 / {len(checkboxes)} (0%)")

# עדכון מד ההתקדמות בזמן אמת
def update_progress(change):
    completed = sum(1 for cb in checkboxes if cb.value)
    progress.value = completed
    percent = int((completed / len(checkboxes)) * 100)
    progress_label.value = f"{completed} / {len(checkboxes)} ({percent}%)"
    
    if completed == len(checkboxes):
        progress.bar_style = 'success'
    else:
        progress.bar_style = 'info'

for cb in checkboxes:
    cb.observe(update_progress, names='value')

# 4. עטיפת כל הווידג'טים בקופסה מימין לשמאל
header_box = widgets.HBox([progress_label, progress])
checklist_box = widgets.VBox([header_box] + ui_elements)

# הוספת מחלקת ה-RTL לקופסה הראשית
checklist_box.add_class("rtl-checklist")

display(checklist_box)

In [17]:
# ספריות
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, LeaveOneOut



In [45]:
db = pd.read_csv('loan_data_v2.csv')
# no null values
display(db.isna().sum())
print(db['person_home_ownership'].unique())
dict_home_ownership = {'RENT': 1, 'OWN': 2, 'MORTGAGE': 3, 'OTHER': 4}
dict_home_ownership
#db2['previous_loan_defaults_on_file'] = db2['previous_loan_defaults_on_file'].map({'No': 0 , 'Yes': 1})
#db2['person_home_ownership'] = db2['person_home_ownership'].map(dict_home_ownership)

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

['OTHER' 'RENT' 'OWN' 'MORTGAGE']


{'RENT': 1, 'OWN': 2, 'MORTGAGE': 3, 'OTHER': 4}

In [46]:
sum(db['loan_status']==1)

2671

In [47]:
db2 = db.copy()
db2 = db2.drop(['person_age','person_gender','cb_person_cred_hist_length','person_education','loan_intent','person_home_ownership'], axis = 1)
db2 = pd.get_dummies(db2, columns=['previous_loan_defaults_on_file'], drop_first=True)
db2

,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,credit_score,loan_status,previous_loan_defaults_on_file_Yes
0,29958.61,13,4923.19,15.32,0.16,608,0,True
1,103953.94,4,35000.00,15.38,0.34,608,1,False
2,51507.44,13,17642.82,16.54,0.34,501,0,True
3,76598.94,13,12694.67,14.87,0.17,633,0,False
4,44134.42,3,12462.46,15.61,0.28,588,0,False
...,...,...,...,...,...,...,...,...
11995,71097.07,23,11086.21,17.18,0.16,620,0,True
11996,26371.24,1,13540.44,16.78,0.51,592,0,True
11997,27586.56,20,7764.84,19.17,0.28,585,0,False
11998,64217.59,10,13853.73,17.88,0.22,553,0,True


In [48]:
db2 = db.copy()
db2 = db2.drop(['person_age','person_gender','cb_person_cred_hist_length','person_education','loan_intent','person_home_ownership'], axis = 1)
db2 = pd.get_dummies(db2, columns=['previous_loan_defaults_on_file'], drop_first=True)#scaler = StandardScaler()

X = db2.drop('loan_status', axis=1).values
y = db2['loan_status'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

C_range = [0.1, 1, 10]
best_score = 0
best_C = None
best_pipeline = None

for c_val in C_range:
    loan_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=c_val))
    ])

    scores = cross_val_score(
        loan_pipeline, 
        X_train, 
        y_train, 
        cv=5, 
        scoring='accuracy'
    )
    mean_score = np.mean(scores)
    print(f"Testing C={c_val} | Mean Accuracy: {mean_score:.4f}")
    
    if mean_score > best_score:
        best_score = mean_score
        best_C = c_val
        best_pipeline = loan_pipeline
print(f"The winning parameter is C={best_C} with an accuracy of {best_score:.4f}")
best_pipeline.fit(X_train, y_train)
y_pred_best = best_pipeline.predict(X_test)
print("\nAccuracy on Test Set:", accuracy_score(y_test, y_pred_best))

cm = confusion_matrix(y_test, y_pred_best)
print("Confusion matrix:\n", cm)
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Classification Report:\n")
print(classification_report(y_test, y_pred_best))

Testing C=0.1 | Mean Accuracy: 0.8545
Testing C=1 | Mean Accuracy: 0.8528
Testing C=10 | Mean Accuracy: 0.8521
The winning parameter is C=0.1 with an accuracy of 0.8545

Accuracy on Test Set: 0.8633333333333333
Confusion matrix:
 [[1726  140]
 [ 188  346]]
Accuracy: 0.8633333333333333
Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.92      0.91      1866
           1       0.71      0.65      0.68       534

    accuracy                           0.86      2400
   macro avg       0.81      0.79      0.80      2400
weighted avg       0.86      0.86      0.86      2400



In [49]:
import joblib

# שמירת ה-Pipeline המאומן לקובץ
joblib.dump(best_pipeline, 'loan_model.pkl')

print("המודל נשמר בהצלחה לקובץ!")

המודל נשמר בהצלחה לקובץ!
